## ⚙️ Configuration
Change the `DATASET_ID` here to switch between FD001, FD002, FD003, or FD004 in one place!

In [ ]:
DATASET_ID = "FD002"  # Options: FD001, FD002, FD003, FD004

TRAIN_FILE = f"train_{DATASET_ID}.txt"
TEST_FILE = f"test_{DATASET_ID}.txt"
RUL_FILE = f"RUL_{DATASET_ID}.txt"

print(f"Targeting Dataset: {DATASET_ID}")
print(f"Train file: {TRAIN_FILE}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
class CMAPSSDataset(Dataset):
    def __init__(self, file_path, sequence_length=30, missing_prob=0.1, noise_level=0.1):
        self.sequence_length = sequence_length
        self.missing_prob = missing_prob
        self.noise_level = noise_level

        columns = ['unit', 'time', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]
        df = pd.read_csv(file_path, sep=r'\s+', names=columns)

        # Standard 14 informative sensors
        self.sensor_cols = ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']

        # Piecewise Linear RUL: Clip at 125
        rul = pd.DataFrame(df.groupby('unit')['time'].max()).reset_index()
        rul.columns = ['unit', 'max']
        df = df.merge(rul, on=['unit'], how='left')
        df['RUL'] = df['max'] - df['time']
        df['RUL'] = df['RUL'].clip(upper=125)
        df.drop('max', axis=1, inplace=True)

        # Operating Condition-Aware Normalization
        df['cond'] = df[['os1', 'os2', 'os3']].round(1).apply(tuple, axis=1)
        for col in self.sensor_cols:
            def cond_min_max(group):
                g_min, g_max = group.min(), group.max()
                if g_max - g_min > 1e-9:
                    return (group - g_min) / (g_max - g_min)
                return group - group
            df[col] = df.groupby('cond')[col].transform(cond_min_max)
        df.drop('cond', axis=1, inplace=True)

        self.sequences = []
        self.labels = []

        for unit_id in df['unit'].unique():
            unit_data = df[df['unit'] == unit_id]
            for i in range(len(unit_data) - sequence_length + 1):
                window = unit_data.iloc[i:i + sequence_length]
                self.sequences.append(window[self.sensor_cols].values)
                self.labels.append(window['RUL'].iloc[-1])

        self.sequences = np.array(self.sequences)
        self.labels = np.array(self.labels)

    def apply_synthetic_faults(self, sensor_data):
        faulty_data = sensor_data.copy()
        for i in range(faulty_data.shape[1]):
            if np.random.rand() < self.missing_prob:
                faulty_data[:, i] = 0.0
            else:
                noise = np.random.normal(0, self.noise_level, size=self.sequence_length)
                faulty_data[:, i] += noise * np.abs(faulty_data[:, i])
        return faulty_data

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        clean_sensors = self.sequences[idx]
        label = torch.tensor([self.labels[idx]], dtype=torch.float32)
        faulty_sensors = self.apply_synthetic_faults(clean_sensors)
        return torch.tensor(faulty_sensors.T, dtype=torch.float32), label

## Architecture

Sensor Encoder has 1D CNN

Weighted Fusion with 3 layers

In [ ]:
class SensorEncoder(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, feature_dim=32):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden_dim, feature_dim, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return torch.mean(x, dim=-1)

class ReliabilityEstimator(nn.Module):
    def __init__(self, feature_dim=32, hidden_dim=16):
        super().__init__()
        self.fc1 = nn.Linear(feature_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
    def forward(self, features):
        x = self.relu(self.fc1(features))
        return self.fc2(x)

class ReliabilityWeightedFusion(nn.Module):
    def __init__(self, num_sensors=14, feature_dim=32):
        super().__init__()
        self.num_sensors = num_sensors
        self.encoders = nn.ModuleList([SensorEncoder(feature_dim=feature_dim) for _ in range(num_sensors)])
        self.reliability_estimators = nn.ModuleList([ReliabilityEstimator(feature_dim=feature_dim) for _ in range(num_sensors)])
        self.predictor = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, sensor_inputs):
        feature_vectors = []
        reliability_logits = []
        for i in range(self.num_sensors):
            h_i = self.encoders[i](sensor_inputs[i].unsqueeze(1))
            feature_vectors.append(h_i)
            reliability_logits.append(self.reliability_estimators[i](h_i))
        weights = F.softmax(torch.stack(reliability_logits, dim=1), dim=1)
        fused_vector = torch.sum(weights * torch.stack(feature_vectors, dim=1), dim=1)
        return self.predictor(fused_vector), weights

## Loads all classes and trains model

In [ ]:
def nasa_scoring_function(y_pred, y_true):
    d = y_pred - y_true
    early_mask, late_mask = d < 0, d >= 0
    early_score = torch.sum(torch.exp(-d[early_mask] / 13.0) - 1)
    late_score = torch.sum(torch.exp(d[late_mask] / 10.0) - 1)
    return (early_score if early_score.numel() > 0 else 0) + (late_score if late_score.numel() > 0 else 0)

def train_model(data_path, epochs=30, batch_size=64, lr=0.001):
    dataset = CMAPSSDataset(data_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ReliabilityWeightedFusion(num_sensors=14).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    mse_criterion = nn.MSELoss()
    print(f"Training on {device}...")
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        epoch_nasa = 0
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)
            sensor_inputs = [data[:, i, :] for i in range(14)]
            optimizer.zero_grad()
            pred, _ = model(sensor_inputs)
            loss = torch.sqrt(mse_criterion(pred, target))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            epoch_nasa += nasa_scoring_function(pred, target).item()
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dataloader):.2f} | NASA: {int(epoch_nasa/len(dataloader)):,}")
    return model

In [ ]:
if os.path.exists(TRAIN_FILE):
    model = train_model(TRAIN_FILE, epochs=30)
else:
    print(f"Error: '{TRAIN_FILE}' not found. Current Path: {os.getcwd()}")

## Evaluate Test Set

In [ ]:
def evaluate_test_set(model, test_path, truth_path, sequence_length=30):
    sensor_cols = ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
    columns = ['unit', 'time', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]
    test_df = pd.read_csv(test_path, sep=r'\s+', names=columns)
    truth_df = pd.read_csv(truth_path, sep=r'\s+', names=['RUL'])
    
    test_df['cond'] = test_df[['os1', 'os2', 'os3']].round(1).apply(tuple, axis=1)
    for col in sensor_cols:
        def cond_min_max(group):
            g_min, g_max = group.min(), group.max()
            if g_max - g_min > 1e-9:
                return (group - g_min) / (g_max - g_min)
            return group - group
        test_df[col] = test_df.groupby('cond')[col].transform(cond_min_max)
    
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    targets = []

    for i, unit_id in enumerate(test_df['unit'].unique()):
        unit_data = test_df[test_df['unit'] == unit_id]
        if len(unit_data) >= sequence_length:
            window = unit_data.iloc[-sequence_length:]
            sensors = window[sensor_cols].values
            data = torch.tensor(sensors.T, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                sensor_inputs = [data[:, j, :] for j in range(14)]
                pred, _ = model(sensor_inputs)
                predictions.append(pred.item())
                targets.append(truth_df.iloc[i]['RUL'])

    predictions = torch.tensor(predictions)
    targets = torch.tensor(targets).clamp(max=125)
    rmse = torch.sqrt(nn.MSELoss()(predictions, targets.float()))
    nasa = nasa_scoring_function(predictions, targets)
    print(f"\n--- Test Results for {DATASET_ID} ---")
    print(f"Test RMSE: {rmse.item():.2f}")
    print(f"Test NASA Score: {int(nasa.item()):,}")
    return rmse.item(), nasa.item()

In [ ]:
if os.path.exists(TEST_FILE) and os.path.exists(RUL_FILE):
    evaluate_test_set(model, TEST_FILE, RUL_FILE)

## Visualize which sensors are ignored and have low reliability score

In [ ]:
def visualize_reliability(model, data_path):
    dataset = CMAPSSDataset(data_path)
    data, target = dataset[0]
    data = data.unsqueeze(0).to(next(model.parameters()).device)
    sensor_inputs = [data[:, i, :] for i in range(14)]
    
    model.eval()
    with torch.no_grad():
        pred, weights = model(sensor_inputs)
        
    weights = weights.squeeze().cpu().numpy()
    sensor_names = ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
    
    plt.figure(figsize=(12, 5))
    plt.bar(sensor_names, weights, color='skyblue')
    plt.title(f"Sensor Reliability ({DATASET_ID}: Pred: {pred.item():.1f}, True: {target.item():.1f})")
    plt.ylabel("Softmax Weight")
    plt.xticks(rotation=45)
    plt.show()

if os.path.exists(TRAIN_FILE):
    visualize_reliability(model, TRAIN_FILE)